In [ ]:
import logging

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = (
    SparkSession.builder 
    .appName("MinIO test")
    .master("local[*]")  
    .config("spark.jars.packages", 
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262") 
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9050") 
    .config("spark.hadoop.fs.s3a.access.key", "adm") # root name
    .config("spark.hadoop.fs.s3a.secret.key", "secret_pass") # root password
    .config("spark.hadoop.fs.s3a.path.style.access", "true") 
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") 
    .getOrCreate()
)

In [ ]:
df = spark.read.parquet("s3a://raw-data/taxi-raw/*.parquet")

In [ ]:
df.printSchema()

### Сделаем выборку без аномалий

In [ ]:
df_filer_one = df.filter(
   (df.trip_miles > 0) & (df.trip_time > 0) & (df.driver_pay > 0) & (df.tips >= 0)
)

### Приводим к желаемому типу

In [ ]:
df_filer_one = df_filer_one.withColumn(
    "request_datetime", F.to_timestamp(F.col("request_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "on_scene_datetime", F.to_timestamp(F.col("on_scene_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "pickup_datetime", F.to_timestamp(F.col("pickup_datetime"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "dropoff_datetime", F.to_timestamp(F.col("dropoff_datetime"), "yyyy-MM-dd HH:mm:ss")
)

In [ ]:
# Добавим еще одну колонку с только с датой поступления запроса
df_filer_one = df_filer_one.withColumn(
    "request_date", F.to_date(F.col("request_datetime"))
)

### Заполним пропуски

In [ ]:
df_filer_one = df_filer_one.fillna({
    "originating_base_num":"UNKWN"   
})  

### Общая плата за проезд

In [ ]:
df_filer_one = df_filer_one.withColumn(
    "total_price", F.col("base_passenger_fare") + F.col("tolls") + F.col("bcf") + F.col("sales_tax") + F.col("congestion_surcharge") + F.col("airport_fee")
)

### Сделаем выбор только определенных призанков

In [ ]:
# Что возьмем для сохранения

#  |-- request_datetime: timestamp_ntz (nullable = true)    --- Дата и время, когда пассажир запросил поезду
#  |-- on_scene_datetime: timestamp_ntz (nullable = true)   --- Дата и время прибытия водителя на место посадки
#  |-- pickup_datetime: timestamp_ntz (nullable = true)     --- Дата и время начала поездки
#  |-- dropoff_datetime: timestamp_ntz (nullable = true)    --- Дата и время высадки
#  |-- PULocationID: integer (nullable = true)              --- ID зоны-посадки
#  |-- DOLocationID: integer (nullable = true)              --- ID зоны-высадки
#  |-- trip_miles: double (nullable = true)                 --- Общее растояние поездки (в милях)
#  |-- total_price: double                                  --- Добавим колонку со всей ценой за поездку
#  |-- tips: double (nullable = true)                       --- Общая сумма чаевых, полученных от пассажира.
#  |-- driver_pay: double (nullable = true)                 --- Общая заработная плата водителя (без учета платы за проезд по платным дорогам и чаевых, а также за вычетом комиссионных, дополнительных сборов и налогов).
#  |-- shared_request_flag: string (nullable = true)        --- Согласился ли пассажир на совместную поездку, независимо от того, были ли они подобраны друг другу? (Да/Нет)
#  |-- shared_match_flag: string (nullable = true)          --- Ехал ли пассажир вместе с другим пассажиром, забронировавшим поездку отдельно

In [ ]:
df_filer_one = df_filer_one.select(
    F.col("request_datetime"),
    F.col("on_scene_datetime"),
    F.col("pickup_datetime"),
    F.col("dropoff_datetime"),
    F.col("PULocationID"),
    F.col("DOLocationID"),
    F.col("trip_miles"),
    F.col("total_price"),
    F.col("tips"),
    F.col("driver_pay"),
    F.col("shared_request_flag"),
    F.col("shared_match_flag"),
    F.col("request_date")
)

In [ ]:
df_filer_one.count()

In [ ]:
df_filer_one.rdd.getNumPartitions()

### Сколько пропусков

In [ ]:
# === Проверка на пропуски ===
df_filer_one.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_filer_one.columns]).show(vertical=True)

In [ ]:
df_filer_one.show(10)

### Запись частями

In [ ]:
date_group = df_filer_one.groupBy(F.col("request_date")).agg(
    F.count("*").alias("row_count")
)

In [ ]:
dates = date_group.collect()

In [ ]:
for item in dates:
    date, row_count = item
    sub_df = df_filer_one.filter(F.col("request_date") == date)
    
    print(str(date))
    sub_df.write.mode("overwrite").parquet(f"s3a://silver-data/tree-part/date-part/{date}") # указать свой путь для сохранения
    break